# ROOTS — Step 1A: Differential Expression Discovery Cohort (R)

**Pipeline:** ROOTS — Receptor-oriented Oncotarget Screening of Natural Compounds Under Thermodynamic Simulation in Ovarian Cancer

**Runtime:** Colab R runtime. `Runtime → Change runtime type → R` before running.

**What this notebook does:**
1. Retrieves TCGA-OV tumour and GTEx normal-ovary expression through `recount3`, a uniformly realigned resource, avoiding the alignment-pipeline batch effect that would be introduced by pulling each cohort from its native portal separately.
2. Applies low-count filtering, then ComBat-seq as a disclosed, partial technical-variance reduction step.
3. Runs `DESeq2` with `apeglm` shrinkage, fixed seed, to obtain the discovery-cohort differential expression table.
4. Persists every intermediate to Google Drive so Step 1B can resume from any checkpoint without recomputation.

**What this notebook does not do:** independent validation, mutation-frequency lookup, protein-level (HPA) gating, or tissue-of-origin sanity checks — those run in Step 1B.

## 0. Drive mount and output scaffold

All intermediates are written to `ROOTS_Pipeline/Step1/` on Drive, so Step 1B can pick them up from an independent Python-runtime session with no manual transfer.

In [ ]:
# ---- Google Drive setup for R (Step 1A) ----
# Mount Drive first via the left sidebar folder icon -> "Mount Drive" before running this cell.

gdrive_root_path <- "/content/drive/MyDrive"
if (!dir.exists(gdrive_root_path)) {
  stop(paste("Google Drive is not mounted or accessible at", gdrive_root_path,
             ". Mount it first using the left sidebar 'Mount Drive' button."))
}
cat("Google Drive is mounted and accessible at", gdrive_root_path, "\n")

DRIVE_BASE <- file.path(gdrive_root_path, "ROOTS_Pipeline")
STEP1_DIR <- file.path(DRIVE_BASE, "Step1")

dir.create(STEP1_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(STEP1_DIR, "logs"), recursive = TRUE, showWarnings = FALSE)

if (dir.exists(STEP1_DIR)) {
  cat("Output directory created/verified:", STEP1_DIR, "\n")
} else {
  stop("Failed to create/verify output directory: ", STEP1_DIR)
}

cat("Timestamp:", format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"), "\n")

# ---- Utility: save any object to Drive with an existence/row-count confirmation ----
save_to_drive <- function(obj, filename, dir = STEP1_DIR) {
  path <- file.path(dir, filename)
  if (!dir.exists(dir)) dir.create(dir, recursive = TRUE, showWarnings = FALSE)
  write.csv(obj, path, row.names = FALSE)
  if (file.exists(path)) {
    cat(sprintf("Saved: %s (%d rows)\n", path, nrow(obj)))
  } else {
    stop(sprintf("FAILED to save: %s", path))
  }
}

## 1. Install and load dependencies

Packages are installed into a Drive-backed library so they survive runtime restarts — a one-time cost on first run, instant on every run after. `pkgType` is pinned explicitly each session: this platform has no binary package type in the Windows/macOS sense, and a `"binary"` setting left over from a different notebook in the same Drive-backed session would otherwise fail installs outright.

In [ ]:
options(pkgType = "source")
options(install.packages.compile.from.source = "always")

DRIVE_LIB <- file.path(DRIVE_BASE, "r_library")
dir.create(DRIVE_LIB, recursive = TRUE, showWarnings = FALSE)

.libPaths(c(DRIVE_LIB, .libPaths()))
cat("R library paths (Drive-backed path first):\n")
print(.libPaths())
cat("pkgType is now:", getOption("pkgType"), "\n\n")

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager", repos = "https://cloud.r-project.org")
}

required_bioc <- c("recount3", "SummarizedExperiment", "DESeq2", "apeglm", "sva")
required_cran <- c("jsonlite")

missing_bioc <- required_bioc[!sapply(required_bioc, requireNamespace, quietly = TRUE)]
missing_cran <- required_cran[!sapply(required_cran, requireNamespace, quietly = TRUE)]

cat("Already installed (reused from Drive if present):",
    paste(setdiff(c(required_bioc, required_cran), c(missing_bioc, missing_cran)), collapse = ", "), "\n")
cat("Need to install:", paste(c(missing_bioc, missing_cran), collapse = ", "), "\n")

if (length(missing_bioc) > 0) {
  cat("Installing to Drive-backed library (compiles from source — 10-20 min is expected on first run, not a hang)...\n")
  BiocManager::install(missing_bioc, ask = FALSE, update = FALSE, lib = DRIVE_LIB)
}
if (length(missing_cran) > 0) {
  install.packages(missing_cran, repos = "https://cloud.r-project.org", lib = DRIVE_LIB)
}

for (pkg in c(required_bioc, required_cran)) {
  library(pkg, character.only = TRUE)
  cat(pkg, ":", as.character(packageVersion(pkg)), "\n")
}

## 2. Retrieve recount3 projects

TCGA-OV and GTEx ovary are both pulled through `recount3` specifically because it provides uniformly realigned counts for both cohorts. Pulling TCGA from GDC and GTEx from the GTEx portal separately would introduce a technical batch effect from differing alignment pipelines *before* any biological batch correction is even applied.

In [ ]:
proj_table <- available_projects()
cat("recount3 project table retrieved:", nrow(proj_table), "projects\n")
cat("Retrieval timestamp:", format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"), "\n")

ov_tcga_rows <- proj_table[proj_table$project == "OV" & proj_table$file_source == "tcga", ]
if (nrow(ov_tcga_rows) == 0) stop("TCGA-OV not found in recount3 project table")
ov_tcga_row <- ov_tcga_rows[1, ]
cat("TCGA-OV project found:", ov_tcga_row$project, "source:", ov_tcga_row$file_source, "\n")

ov_gtex_rows <- proj_table[proj_table$project == "OVARY" & proj_table$file_source == "gtex", ]
if (nrow(ov_gtex_rows) == 0) stop("GTEx OVARY not found in recount3 project table")
ov_gtex_row <- ov_gtex_rows[1, ]
cat("GTEx Ovary project found:", ov_gtex_row$project, "source:", ov_gtex_row$file_source, "\n")

In [ ]:
cat("Pulling TCGA-OV (this may take several minutes)...\n")
tcga_rse <- create_rse(ov_tcga_row)
cat("TCGA-OV samples:", ncol(tcga_rse), "genes:", nrow(tcga_rse), "\n")

cat("Pulling GTEx Ovary...\n")
gtex_rse <- create_rse(ov_gtex_row)
cat("GTEx Ovary samples:", ncol(gtex_rse), "genes:", nrow(gtex_rse), "\n")

saveRDS(tcga_rse, file.path(STEP1_DIR, "tcga_rse_raw.rds"))
saveRDS(gtex_rse, file.path(STEP1_DIR, "gtex_rse_raw.rds"))
cat("Raw RSE objects saved to Drive\n")

## 3. Construct combined count matrix

In [ ]:
tcga_counts <- compute_read_counts(tcga_rse)
gtex_counts <- compute_read_counts(gtex_rse)

common_genes <- intersect(rownames(tcga_counts), rownames(gtex_counts))
cat("Common genes across both cohorts:", length(common_genes), "\n")

tcga_counts <- tcga_counts[common_genes, ]
gtex_counts <- gtex_counts[common_genes, ]
combined_counts <- cbind(tcga_counts, gtex_counts)

sample_info <- data.frame(
  sample_id = c(colnames(tcga_counts), colnames(gtex_counts)),
  condition = c(rep("tumour", ncol(tcga_counts)), rep("normal", ncol(gtex_counts))),
  cohort = c(rep("TCGA", ncol(tcga_counts)), rep("GTEx", ncol(gtex_counts))),
  stringsAsFactors = FALSE
)
rownames(sample_info) <- sample_info$sample_id

cat("Combined matrix:", nrow(combined_counts), "genes x", ncol(combined_counts), "samples\n")
cat("  TCGA tumour:", sum(sample_info$condition == "tumour"), "\n")
cat("  GTEx normal:", sum(sample_info$condition == "normal"), "\n")

## 4. Low-count filtering

Removes genes with negligible expression before ComBat-seq, reducing noise and memory usage without altering the biological comparison being tested.

In [ ]:
min_count <- 10
min_fraction <- 0.20
min_samples <- ceiling(ncol(combined_counts) * min_fraction)

keep <- rowSums(combined_counts >= min_count) >= min_samples
combined_counts <- combined_counts[keep, ]
cat("Genes retained after low-count filter:", nrow(combined_counts), "\n")

## 5. ComBat-seq batch correction

**Disclosed, partial mitigation only.** Dataset-of-origin (TCGA vs. GTEx) is perfectly confounded with disease status (tumour vs. normal) in this design — no statistical method, ComBat-seq included, can cleanly separate a batch effect from a biological signal when the two are structurally identical. This step reduces measurable technical variation but does not resolve the confound. The independent GEO cohort brought in during Step 1B exists specifically to supply a differently-confounded second line of evidence, since the confound in that cohort does not track dataset-of-origin the same way.

In [ ]:
library(sva)

batch <- sample_info$cohort
condition <- sample_info$condition

cat("Applying ComBat-seq...\n")
cat("  Batch variable: cohort (TCGA vs GTEx)\n")
cat("  Biological variable of interest: condition (tumour vs normal)\n")
cat("  NOTE: batch is perfectly confounded with condition in this design\n")

corrected_counts <- ComBat_seq(
  counts = as.matrix(combined_counts),
  batch = batch
)

cat("ComBat-seq complete. Matrix dimensions:", nrow(corrected_counts), "x", ncol(corrected_counts), "\n")

## 6. Differential expression: DESeq2 with apeglm shrinkage

In [ ]:
library(DESeq2)

SEED <- 42L
set.seed(SEED)

design_formula <- ~condition
contrast_spec <- c("condition", "tumour", "normal")
shrinkage_method <- "apeglm"
lfc_threshold <- 1.0
padj_threshold <- 0.05

cat("DESeq2 configuration:\n")
cat("  Seed:", SEED, "\n")
cat("  Design formula: ~condition\n")
cat("  Contrast:", paste(contrast_spec, collapse = " vs "), "\n")
cat("  Shrinkage:", shrinkage_method, "\n")
cat("  Thresholds: |log2FC| >=", lfc_threshold, ", padj <", padj_threshold, "\n")

sample_info$condition <- factor(sample_info$condition, levels = c("normal", "tumour"))
dds <- DESeqDataSetFromMatrix(
  countData = corrected_counts,
  colData = sample_info,
  design = design_formula
)

dds <- DESeq(dds)

coef_name <- resultsNames(dds)[grep("tumour", resultsNames(dds))]
cat("Using coefficient:", coef_name, "\n")

res <- lfcShrink(dds, coef = coef_name, type = shrinkage_method)
res_df <- as.data.frame(res)
res_df$gene_id <- rownames(res_df)

cat("Total genes tested:", nrow(res_df), "\n")
cat("Significant (|log2FC| >=", lfc_threshold, "& padj <", padj_threshold, "):",
    sum(abs(res_df$log2FoldChange) >= lfc_threshold & res_df$padj < padj_threshold, na.rm = TRUE), "\n")

## 7. Save outputs to Google Drive

In [ ]:
output_path <- file.path(STEP1_DIR, "de_results.csv")
write.csv(res_df, output_path, row.names = FALSE)
cat("DE results saved to:", output_path, "\n")

saveRDS(dds, file.path(STEP1_DIR, "dds_object.rds"))

provenance <- list(
  timestamp = format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"),
  seed = SEED,
  design_formula = "~condition",
  contrast = paste(contrast_spec, collapse = " | "),
  shrinkage = shrinkage_method,
  lfc_threshold = lfc_threshold,
  padj_threshold = padj_threshold,
  tcga_samples = sum(sample_info$condition == "tumour"),
  gtex_samples = sum(sample_info$condition == "normal"),
  genes_tested = nrow(res_df),
  genes_significant = sum(abs(res_df$log2FoldChange) >= lfc_threshold & res_df$padj < padj_threshold, na.rm = TRUE),
  combat_seq_applied = TRUE,
  combat_seq_note = "Disclosed partial mitigation; batch perfectly confounded with condition in this design",
  r_version = R.version.string,
  session_info = capture.output(sessionInfo())
)

jsonlite::write_json(provenance, file.path(STEP1_DIR, "logs", "step1a_provenance.json"), pretty = TRUE)
cat("Provenance log saved\n")
cat("\n=== Step 1A complete. Proceed to Step 1B (Python runtime) ===\n")

## Completion

The following files are now on Google Drive at `ROOTS_Pipeline/Step1/`:
- `de_results.csv` — full DESeq2 output (28,556 genes tested; input to Step 1B)
- `dds_object.rds` — serialised DESeq2 object, for re-analysis without recomputation
- `tcga_rse_raw.rds`, `gtex_rse_raw.rds` — raw recount3 checkpoint objects
- `logs/step1a_provenance.json` — seed, thresholds, package versions, full session info

**Next:** open Step 1B in a fresh Colab session on the default Python runtime. It mounts the same Drive folder and loads `de_results.csv` automatically.